# Global Fourier, Winsorized Doppler & Coherent P-FFA Orbit Fitting

Demonstrates `pyloki.orbit_fit`:
1. `fit_phase_curve_fourier`: global linear least-squares circular/eccentric orbit discovery with eclipse masking.
2. `fit_keplerian_multistart`: exact 5-parameter Keplerian phase-delay polisher seeded from `FourierOrbitFit`.
3. `fit_keplerian_periods_winsorized`: MAD-scaled outlier-resistant Doppler frequency fitter (15% corrupted segments).

In [1]:
import math

import numpy as np

from pyloki import kepler, orbit_fit
from pyloki.utils.misc import C_VAL

# True eccentric binary parameters
f0_true, p_orb_true, x_true, ecc_true, aop_true, phi_true = (
    200.0,
    3000.0,
    1.8,
    0.25,
    0.9,
    0.6,
)
t = np.linspace(0.0, 6000.0, 100, dtype=np.float64)
z_m, v_ms, _ = kepler.keplerian_z_derivatives_exact(
    t, p_orb_true, ecc_true, phi_true, x_true * C_VAL, aop_true, 0.5 * math.pi
)
delays = 0.2 + 1.5e-6 * t + z_m / C_VAL

# Mask out a 15% eclipsed region and corrupt it
mask = np.ones(t.size, dtype=np.bool_)
mask[35:50] = False
delays_obs = delays.copy()
delays_obs[~mask] += 25.0

In [2]:
# 1. Global Fourier phase-curve fit
fourier_fit = orbit_fit.fit_phase_curve_fourier(
    t,
    delays_obs,
    p_orb_range=(2200.0, 4000.0),
    fit_eccentric=True,
    n_harmonics=6,
    fit_mask=mask,
)
print(
    f"Fourier fit: P_orb={fourier_fit.p_orb:.2f} s, x={fourier_fit.x_orb:.3f} lt-s, "
    f"e_approx={fourier_fit.ecc_approx:.3f}, RMS={fourier_fit.rms_residual:.3e} s"
)

# 2. Multi-start exact Keplerian polisher
kep_fit = orbit_fit.fit_keplerian_multistart(
    t,
    delays_obs,
    fourier_seed=fourier_fit,
    fit_mask=mask,
    n_restarts=5,
    seed=11,
)
print(
    f"Keplerian polish: P_orb={kep_fit.p_orb:.2f} s, x={kep_fit.x_orb:.3f} lt-s, "
    f"e={kep_fit.ecc:.3f}, RMS={kep_fit.rms_residual:.3e} s"
)

Fourier fit: P_orb=2999.99 s, x=1.747 lt-s, e_approx=0.246, RMS=9.694e-05 s
Keplerian polish: P_orb=3000.00 s, x=1.800 lt-s, e=0.250, RMS=4.390e-12 s


In [3]:
# 3. Outlier-resistant Winsorized Doppler frequency fit (15% corrupted segments)
rng = np.random.default_rng(42)
freqs_obs = f0_true * (1.0 - v_ms / C_VAL)
bad = rng.choice(t.size, size=15, replace=False)
freqs_obs[bad] += rng.choice([-0.5, 0.5], size=15)

win_fit = orbit_fit.fit_keplerian_periods_winsorized(
    t,
    freqs_obs,
    initial_guess=[
        f0_true * 1.00001,
        p_orb_true * 1.01,
        x_true * 0.96,
        ecc_true + 0.03,
        aop_true + 0.1,
        phi_true - 0.1,
    ],
    n_restarts=4,
    seed=7,
)
print(
    f"Winsorized Doppler fit: f0={win_fit.f0:.5f} Hz, P_orb={win_fit.p_orb:.2f} s, "
    f"x={win_fit.x_orb:.3f} lt-s, e={win_fit.ecc:.3f}"
)

Winsorized Doppler fit: f0=200.00227 Hz, P_orb=3009.71 s, x=1.810 lt-s, e=0.245
